# 01 — Desviación de lluvias: campaña 2022/23

**Evento:** sequía 2022/23, la peor en ~60 años (~USD 19.000 millones de pérdida según BCR).

**Pregunta:** ¿cuánto llovió en la campaña gruesa (septiembre 2022 → marzo 2023) respecto de lo normal, partido por partido?

**Datos:** CHIRPS Pentad (`UCSB-CHG/CHIRPS/PENTAD`, ~5,5 km, 1981–presente). Línea base: media de las campañas 1981/82 → 2020/21 (40 campañas). Límites administrativos: FAO GAUL nivel 2.

**Salidas** (exportadas a `assets/`):
1. Mapa de anomalía relativa de precipitación por partido (zona agrícola: Buenos Aires, Córdoba, Santa Fe, Entre Ríos, La Pampa).
2. Ranking de partidos más secos.
3. Serie mensual de Pergamino (partido testigo) vs. su climatología.

**Requisitos:** `earthengine authenticate` ya corrido y `EE_PROJECT` seteado en el entorno.

Metodología completa en [docs/METODOLOGIA.md](../docs/METODOLOGIA.md).

In [ ]:
import sys

sys.path.append("../src")

import ee
import geemap
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm

import gee_utils as gu

gu.init_ee()

CAMPAIGN_YEAR = 2022          # campaña sep-2022 -> mar-2023
BASELINE = (1981, 2020)       # campañas 1981/82 -> 2020/21
PROVINCES = ["Buenos Aires", "Cordoba", "Santa Fe", "Entre Rios", "La Pampa"]

## 1. Precipitación de la campaña vs. climatología

Todo server-side: la suma de la campaña, la media histórica de 40 campañas y las anomalías se calculan en GEE; recién bajamos números al agregar por partido.

In [ ]:
partidos = gu.get_partidos(PROVINCES)

season = gu.chirps_season_sum(CAMPAIGN_YEAR)
clim = gu.chirps_season_climatology(*BASELINE)
anom_mm, anom_pct = gu.anomaly_images(season, clim)

stats_img = season.addBands([clim, anom_mm, anom_pct])
fc = gu.zonal_stats(stats_img, partidos, scale=gu.SCALE_CHIRPS)

In [ ]:
# Una sola bajada de datos: FeatureCollection -> GeoDataFrame
gdf = geemap.ee_to_gdf(fc)
gdf = gdf.rename(columns={"ADM1_NAME": "provincia", "ADM2_NAME": "partido"})
cols = ["partido", "provincia", "precip_mm", "precip_mm_mean", "anom_mm", "anom_pct", "geometry"]
gdf = gdf[cols]
print(f"{len(gdf)} partidos/departamentos")
gdf.head(3)

## 2. Mapa: anomalía relativa por partido

Paleta divergente rojo–blanco–azul centrada en 0 (convención del proyecto: rojo = déficit, azul = exceso).

In [ ]:
cmap = LinearSegmentedColormap.from_list("anom", gu.PALETTES["anomaly"])
vmax = float(gdf["anom_pct"].abs().quantile(0.98))
norm = TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)

fig, ax = plt.subplots(figsize=(9, 11))
gdf.plot(column="anom_pct", cmap=cmap, norm=norm, ax=ax,
         edgecolor="white", linewidth=0.3)
ax.set_axis_off()
ax.set_title(
    "¿Cuánto menos llovió en la campaña 2022/23?\n"
    "Anomalía de precipitación sep-2022 → mar-2023 vs. media 1981–2021",
    fontsize=14, loc="left",
)
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
cbar = fig.colorbar(sm, ax=ax, shrink=0.5, label="Anomalía de precipitación (%)")
ax.annotate(
    "Datos: CHIRPS Pentad (UCSB-CHG) · Límites: FAO GAUL · Elaboración propia con Google Earth Engine",
    xy=(0, 0.01), xycoords="axes fraction", fontsize=8, color="gray",
)
fig.tight_layout()
fig.savefig("../assets/01_anomalia_lluvia_partidos.png", dpi=200, bbox_inches="tight")
plt.show()

## 3. Ranking: los partidos más secos

In [ ]:
ranking = (
    gdf.drop(columns="geometry")
    .assign(anom_pct=lambda d: d.anom_pct.round(1),
            anom_mm=lambda d: d.anom_mm.round(0),
            precip_mm=lambda d: d.precip_mm.round(0),
            precip_mm_mean=lambda d: d.precip_mm_mean.round(0))
    .sort_values("anom_pct")
    .reset_index(drop=True)
)
ranking.to_csv("../data/ranking_anomalia_lluvia_2223.csv", index=False)
ranking.head(15)

## 4. Serie mensual: Pergamino vs. su climatología

In [ ]:
testigo = partidos.filter(ee.Filter.eq("ADM2_NAME", "Pergamino")).geometry()

serie = geemap.ee_to_df(gu.monthly_precip_series(testigo, CAMPAIGN_YEAR))
clim_m = geemap.ee_to_df(gu.monthly_precip_climatology(testigo, *BASELINE))

orden = [9, 10, 11, 12, 1, 2, 3]
serie = serie.set_index("month").loc[orden]
clim_m = clim_m.set_index("month").loc[orden]
labels = ["Sep", "Oct", "Nov", "Dic", "Ene", "Feb", "Mar"]

fig, ax = plt.subplots(figsize=(9, 5))
x = range(len(orden))
ax.bar(x, clim_m["precip_mm_mean"], color="#d1e5f0", label="Media 1981–2021")
ax.bar(x, serie["precip_mm"], width=0.5, color="#b2182b", label="Campaña 2022/23")
ax.set_xticks(list(x), labels)
ax.set_ylabel("Precipitación mensual (mm)")
ax.set_title("Pergamino: la campaña 2022/23 contra 40 años de historia", loc="left")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
ax.annotate("Datos: CHIRPS Pentad · Elaboración propia con Google Earth Engine",
            xy=(0, -0.12), xycoords="axes fraction", fontsize=8, color="gray")
fig.tight_layout()
fig.savefig("../assets/01_serie_pergamino.png", dpi=200, bbox_inches="tight")
plt.show()

## Lecturas

- El mapa deja ver el gradiente de la sequía: el déficit no fue uniforme — eso es exactamente lo que un índice por partido/lote captura y un promedio provincial esconde.
- El ranking alimenta la validación de fase 5: contrastar contra los 68 partidos bonaerenses en emergencia (Res. MEC 587/2023).
- **Siguiente notebook (02):** la lluvia es la causa; el NDVI mide el efecto sobre el cultivo, lote por lote.